<div style="background: linear-gradient(135deg, #090d16, #1e293b); border: 3px solid #3b82f6; border-radius: 20px; padding: 28px 20px; text-align: center; color: white; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; box-shadow: 0 15px 35px rgba(0,0,0,0.4); max-width: 800px; margin: 10px auto;">
  <div style="font-size: 42px; margin-bottom: 6px;">🎭 FreeFakeStudio</div>
  <div style="display: inline-block; background: #2563eb; color: white; font-size: 13px; font-weight: 800; padding: 5px 16px; border-radius: 50px; margin-bottom: 16px; letter-spacing: 0.5px;">
    100% FREE · ZERO CODING · 1-STEP RUN
  </div>
  <h1 style="color: #60a5fa; font-size: 32px; font-weight: 900; margin: 0 0 10px 0; letter-spacing: -0.02em;">
    👇 CLICK THE ROUND PLAY BUTTON ( ▶ ) BELOW 👇
  </h1>
  <p style="color: #cbd5e1; font-size: 17px; margin: 0 0 16px 0; line-height: 1.5;">
    That's all! Just click <b>▶</b> on the left of the box below.<br>
    Your free GPU will start automatically in ~1 minute, and a big button will appear to open your studio!
  </p>
  <div style="font-size: 36px; line-height: 1;">⬇️ &nbsp; ⬇️ &nbsp; ⬇️</div>
</div>

In [ ]:
# @title 🟢 CLICK PLAY HERE ▶️ (START FREE GPU)
# @markdown *(Optional: change model below or just keep the fastest default)*
Model = "⚡ Z-Image Turbo (Fastest ~1 min)"  # @param ["⚡ Z-Image Turbo (Fastest ~1 min)", "🌊 FLUX.2-klein 4B", "🔮 FLUX.2-klein 9B", "🎨 Qwen-Image-Edit", "🖌️ ERNIE-Image Turbo"]

import os, sys, subprocess, time, re, glob
import torch
from IPython.display import display, HTML, clear_output

# ── 1. Check GPU ──────────────────────────────────────────
if not torch.cuda.is_available():
    clear_output()
    display(HTML('''
    <div style="background: #7f1d1d; border: 3px solid #ef4444; border-radius: 16px; padding: 25px; text-align: center; color: white; font-family: sans-serif;">
      <h2 style="margin: 0 0 10px 0; font-size: 24px;">⚠️ GPU Not Connected Yet!</h2>
      <p style="font-size: 16px; margin: 0 0 15px 0;">Please enable Google's free T4 GPU in 2 clicks:</p>
      <ol style="text-align: left; display: inline-block; font-size: 15px; line-height: 1.8;">
        <li>Click <b>Runtime</b> in the top menu</li>
        <li>Click <b>Change runtime type</b></li>
        <li>Choose <b>T4 GPU</b> and click <b>Save</b></li>
        <li>Click <b>▶ Play</b> again!</li>
      </ol>
    </div>
    '''))
    raise SystemExit("GPU required to run FreeFakeStudio")

gpu_name = torch.cuda.get_device_name(0)
print(f"✅ Connected to {gpu_name}!")
print("⏳ Step 1/3: Setting up free AI engine in background...")

# ── 2. ComfyUI & Deps ─────────────────────────────────────
need_gguf = any(k in Model for k in ["Qwen", "9B", "ERNIE"])
need_diffusers = "Qwen" in Model

if not os.path.exists('/content/ComfyUI'):
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", "/content/ComfyUI"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/ComfyUI/requirements.txt"], check=True)

if need_gguf and not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    subprocess.run(["git", "clone", "https://github.com/city96/ComfyUI-GGUF.git", "/content/ComfyUI/custom_nodes/ComfyUI-GGUF"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt"], check=True)

if need_diffusers:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/huggingface/diffusers"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rembg[gpu]", "onnxruntime-gpu", "aria2", "gradio"], check=True)

# ── 3. Model Dirs & Downloads ─────────────────────────────
DIFF   = '/content/ComfyUI/models/diffusion_models'
CLIP   = '/content/ComfyUI/models/clip'
TXTENC = '/content/ComfyUI/models/text_encoders'
VAE    = '/content/ComfyUI/models/vae'
for d in [DIFF, CLIP, TXTENC, VAE]:
    os.makedirs(d, exist_ok=True)

def dl(url, dest_dir, name):
    path = os.path.join(dest_dir, name)
    if os.path.exists(path) and os.path.getsize(path) > 1024:
        print(f'  ✅ {name} (ready)')
    else:
        if os.path.exists(path):
            os.remove(path)
        print(f'  ⏳ Downloading {name}...')
        subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", url, "-d", dest_dir, "-o", name], check=True)
        print(f'  ✅ {name} ready!')

print(f"\n📦 Step 2/3: Downloading model weights for {Model}...")

if "Z-Image Turbo" in Model:
    dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors', DIFF, 'z-image-turbo-fp8-e4m3fn.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors', CLIP, 'qwen_3_4b.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors', VAE, 'ae.safetensors')

elif "FLUX.2-klein 4B" in Model:
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-4B/resolve/main/flux-2-klein-4b.safetensors', DIFF, 'flux-2-klein-4b.safetensors')
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b_fp4_flux2.safetensors', TXTENC, 'qwen_3_4b_fp4_flux2.safetensors')

elif "FLUX.2-klein 9B" in Model:
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-kv-fp8/resolve/main/flux-2-klein-9b-kv-fp8.safetensors', DIFF, 'flux-2-klein-9b-kv-fp8.safetensors')
    dl('https://huggingface.co/unsloth/Qwen3-8B-GGUF/resolve/main/Qwen3-8B-Q2_K_L.gguf', TXTENC, 'Qwen3-8B-Q2_K_L.gguf')

elif "Qwen-Image-Edit" in Model:
    dl('https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/resolve/main/qwen-image-edit-2511-Q3_K_M.gguf', DIFF, 'qwen-image-edit-2511-Q3_K_M.gguf')
    dl('https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/Qwen2.5-VL-7B-Instruct-Q3_K_M.gguf', CLIP, 'Qwen2.5-VL-7B-Instruct-Q3_K_M.gguf')
    dl('https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/mmproj-F16.gguf', CLIP, 'Qwen2.5-VL-7B-Instruct-mmproj-F16.gguf')
    dl('https://huggingface.co/Qwen/Qwen-Image-Edit-2511/resolve/main/vae/diffusion_pytorch_model.safetensors', VAE, 'qwen_image_vae.safetensors')

elif "ERNIE-Image Turbo" in Model:
    dl('https://huggingface.co/unsloth/ERNIE-Image-Turbo-GGUF/resolve/main/ernie-image-turbo-Q6_K.gguf', DIFF, 'ernie-image-turbo-Q6_K.gguf')
    dl('https://huggingface.co/Comfy-Org/ERNIE-Image/resolve/main/text_encoders/ministral-3-3b.safetensors', TXTENC, 'ministral-3-3b.safetensors')
    dl('https://huggingface.co/Comfy-Org/ERNIE-Image/resolve/main/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')

# Symlink clip/ <-> text_encoders/
for f in glob.glob(f'{CLIP}/*'):
    link = os.path.join(TXTENC, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)
for f in glob.glob(f'{TXTENC}/*'):
    link = os.path.join(CLIP, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)

# ── 4. Clone / Pull FreeFakeStudio ────────────────────────
print("\n🚀 Step 3/3: Starting FreeFakeStudio Engine...")
REPO = 'https://github.com/MuntasirMalek/FreeFakeStudio.git'
if os.path.exists('/content/FreeFakeStudio'):
    os.chdir('/content/FreeFakeStudio')
    subprocess.run(["git", "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO, "/content/FreeFakeStudio"], check=True)
    os.chdir('/content/FreeFakeStudio')

# ── 5. Run app.py and capture live link ────────────────────
proc = subprocess.Popen(
    [sys.executable, "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

gradio_url = None
url_pattern = re.compile(r'(https://[a-zA-Z0-9-]+\.gradio\.live)')

for line in proc.stdout:
    print(line, end="")
    match = url_pattern.search(line)
    if match:
        gradio_url = match.group(1)
        break

if gradio_url:
    web_app_url = f"https://muntasirmalek.github.io/FreeFakeStudio/?backend={gradio_url}"
    clear_output()
    display(HTML(f'''
    <div style="background: linear-gradient(135deg, #064e3b, #047857); border: 4px solid #10b981; border-radius: 24px; padding: 42px 24px; text-align: center; color: white; max-width: 680px; margin: 25px auto; box-shadow: 0 25px 60px rgba(0,0,0,0.6); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
        <div style="font-size: 64px; margin-bottom: 12px;">🎉</div>
        <h1 style="color: #ffffff; margin: 0 0 12px 0; font-size: 36px; font-weight: 900; letter-spacing: -0.02em;">
            YOUR GPU IS READY!
        </h1>
        <p style="color: #d1fae5; font-size: 17px; margin: 0 0 28px 0; line-height: 1.6;">
            Keep this Colab tab open in the background.<br>
            Click the huge button below to start creating:
        </p>
        <a href="{web_app_url}" target="_blank" style="display: inline-block; background: #ffffff; color: #047857; padding: 22px 55px; border-radius: 18px; font-size: 26px; font-weight: 900; text-decoration: none; box-shadow: 0 10px 30px rgba(0,0,0,0.35); text-transform: uppercase; letter-spacing: 0.5px;">
            🚀 OPEN STUDIO NOW 🚀
        </a>
        <div style="margin-top: 26px; padding-top: 18px; border-top: 1px solid rgba(255,255,255,0.25); font-size: 14px; color: #a7f3d0;">
            Direct Tunnel: <a href="{gradio_url}" target="_blank" style="color: #ffffff; text-decoration: underline;">{gradio_url}</a>
        </div>
    </div>
    '''))

# Keep process running
try:
    for line in proc.stdout:
        pass
except KeyboardInterrupt:
    proc.terminate()
    print("\n🛑 Engine stopped.")
